# HCC behavioral analysis - readable version

This notebook analyzes HCC behavioral CSV files one run at a time.

## Analysis rules
- The first 2 trials of every run are removed.
- RT uses only correct trials with finite, positive response times. RT is converted from seconds to milliseconds.
- Accuracy uses every remaining trial. A `subject_correct` value of 1 is correct; 0 or missing is incorrect.
- Subject summaries pool trials across runs. Group means give every subject equal weight.

Run the notebook from top to bottom. Routine changes belong in the Settings cell.

In [ ]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# -------------------- Settings --------------------
# Folder containing files such as sub-11113_task-HCC_block-002_date-20260918_1200.csv.
DATA_ROOT = Path('/data/backed_up/shared/ME_7T_Pilot/behav_data')

# Example: SELECTED_SUBJECT = '11113' and SELECTED_RUN = 3.
SELECTED_SUBJECT = '11116'
SELECTED_RUN = 1
SUBJECTS = None  # None = all subjects; or use ['11084', '11113'].

# Example: {'11113': [1, 2]} excludes runs 1 and 2 for subject 11113.
EXCLUDE_RUNS = {'11113': [1, 2]}
N_INITIAL_TRIALS_TO_DROP = 2
REQUIRE_ALL_12_RUNS = False  # True affects only the group mean.

# Use this only when duplicate files exist for the same subject and run.
# Example: FILE_OVERRIDES = {('11113', 3): 'exact_filename.csv'}
FILE_OVERRIDES = {}

SAVE_RESULTS = False
OUTPUT_DIR = Path.cwd() / 'hcc_results_readable'
PANEL_COLUMNS = 3

RUNS = list(range(1, 13))
TRIAL_TYPES = ['EDS', 'IDS', 'Stay']
COLORS = ['#4878D0', '#EE854A', '#6ACC64']
METRICS = [('mean_RT_ms', 'Reaction time (ms)'), ('accuracy_pct', 'Accuracy (%)')]
plt.rcParams.update({'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False})

## 1. Input CSV columns

Each CSV must contain these columns:

| Column | Used for | Expected data |
|---|---|---|
| `trial` | Removing the first trials | integers 1 to 38 |
| `hierarhical_switch_type` | Condition label | `EDS`, `IDS`, or `Stay` |
| `subject_correct` | Accuracy and RT filtering | 1 = correct, 0 or missing = incorrect |
| `response_time` | RT | seconds; positive finite values are usable |

The filename supplies the subject and run. The filename block is zero-based, so `run = block + 1`.

In [ ]:
FILE_PATTERN = re.compile(
    r'^sub-(?P<subject>.+)_task-HCC_block-(?P<block>\d+)_date-(?P<date>\d{8}_\d{4})\.csv$'
)


def discover_files():
    """Find included files and report missing or excluded runs."""
    if not DATA_ROOT.is_dir():
        raise FileNotFoundError(f'Folder not found: {DATA_ROOT}')

    records = []
    for path in sorted(DATA_ROOT.rglob('sub-*_task-HCC_block-*_date-*.csv')):
        match = FILE_PATTERN.fullmatch(path.name)
        if match is None:
            continue
        subject = match['subject']
        run = int(match['block']) + 1
        if run in RUNS and (SUBJECTS is None or subject in SUBJECTS):
            records.append({'subject': subject, 'run': run, 'date': match['date'], 'path': path})

    found = pd.DataFrame(records, columns=['subject', 'run', 'date', 'path'])
    subjects = sorted(set(found['subject']) | set(SUBJECTS or []))
    included, excluded, coverage_rows = [], [], []

    for subject in subjects:
        files = found.loc[found['subject'] == subject]
        present = set(files['run'])
        excluded_runs = set(EXCLUDE_RUNS.get(subject, []))
        coverage_rows.append({
            'subject': subject,
            'n_runs': len(present - excluded_runs),
            'included_runs': sorted(present - excluded_runs),
            'missing_runs': sorted(set(RUNS) - present),
            'excluded_runs': sorted(excluded_runs),
        })

        for run, candidates in files.groupby('run'):
            if run in excluded_runs:
                excluded.extend(candidates.to_dict('records'))
                continue
            override = FILE_OVERRIDES.get((subject, run))
            if override is not None:
                candidates = candidates.loc[candidates['path'].map(lambda p: p.name == override)]
            if len(candidates) != 1:
                raise ValueError(f'{subject}, run {run}: expected one file; found {len(candidates)}')
            included.extend(candidates.to_dict('records'))

    coverage = pd.DataFrame(coverage_rows)
    if not included:
        raise ValueError('No included CSV files were found.')
    manifest = pd.DataFrame(included).sort_values(['subject', 'run']).reset_index(drop=True)
    excluded_files = pd.DataFrame(excluded, columns=found.columns)
    return manifest, excluded_files, coverage


manifest, excluded_files, coverage = discover_files()
print(f'Found {len(manifest)} included runs from {manifest.subject.nunique()} subjects.')
display(coverage)
if not excluded_files.empty:
    display(excluded_files)

## 2. Load and clean trials

For every included CSV, this cell:
1. checks the required columns;
2. converts numeric columns to numbers;
3. removes the first 2 trials within that run;
4. creates `correct`; and
5. creates `RT_ms` only when the response was correct and the original response time was finite and positive.

`data` contains one row per retained trial.

In [ ]:
REQUIRED_COLUMNS = {'trial', 'hierarhical_switch_type', 'subject_correct', 'response_time'}


def load_trials(manifest):
    frames = []
    for row in manifest.itertuples(index=False):
        frame = pd.read_csv(row.path)
        missing = REQUIRED_COLUMNS - set(frame.columns)
        if missing:
            raise ValueError(f'{row.path.name}: missing columns {missing}')

        for column in ['trial', 'subject_correct', 'response_time']:
            frame[column] = pd.to_numeric(frame[column], errors='coerce')
        if not frame['trial'].isin(range(1, 39)).all() or frame['trial'].duplicated().any():
            raise ValueError(f'{row.path.name}: trial must contain unique values from 1 to 38')
        if not frame['subject_correct'].dropna().isin([0, 1]).all():
            raise ValueError(f'{row.path.name}: subject_correct must be 0, 1, or missing')

        frame['trial_type'] = frame['hierarhical_switch_type'].astype('string').str.strip()
        if not frame['trial_type'].isin(TRIAL_TYPES).all():
            raise ValueError(f'{row.path.name}: unexpected trial type')

        # Trial numbers restart in every file, so remove early trials after loading each file.
        frame = frame.loc[frame['trial'] > N_INITIAL_TRIALS_TO_DROP].copy()
        frame['correct'] = frame['subject_correct'].eq(1)
        valid_time = np.isfinite(frame['response_time']) & frame['response_time'].gt(0)
        # response_time is in seconds; multiply by 1000 to store milliseconds.
        frame['RT_ms'] = (1000 * frame['response_time']).where(valid_time & frame['correct'])
        frames.append(frame.assign(subject=row.subject, run=row.run, source_file=row.path.name))

    return pd.concat(frames, ignore_index=True)


data = load_trials(manifest)
display(data[['subject', 'run', 'trial', 'trial_type', 'subject_correct', 'correct', 'response_time', 'RT_ms']].head())

## 3. Calculate run and subject summaries

For each subject/run/condition or subject/condition:
- `n_trials`: number of retained trials, used as the accuracy denominator;
- `n_correct`: number of trials where `subject_correct == 1`;
- `n_rt`: number of valid correct-trial RTs;
- `mean_RT_ms`, `median_RT_ms`, `SD_RT_ms`: calculated from `RT_ms`;
- `accuracy_pct = 100 * n_correct / n_trials`.

In [ ]:
def summarize(data, group_columns):
    summary = data.groupby(group_columns + ['trial_type'], observed=True).agg(
        n_trials=('correct', 'size'),
        n_correct=('correct', 'sum'),
        n_rt=('RT_ms', 'count'),
        mean_RT_ms=('RT_ms', 'mean'),
        median_RT_ms=('RT_ms', 'median'),
        SD_RT_ms=('RT_ms', 'std'),
    ).reset_index()
    summary['accuracy_pct'] = 100 * summary['n_correct'] / summary['n_trials']
    return summary


run_summary = summarize(data, ['subject', 'run'])
subject_summary = summarize(data, ['subject'])
eligible_subjects = coverage.loc[coverage['n_runs'].eq(12), 'subject'] if REQUIRE_ALL_12_RUNS else coverage['subject']
group_subject_summary = subject_summary.loc[subject_summary['subject'].isin(eligible_subjects)].copy()

# Group means average subject means, so a subject with more runs does not get extra weight.
group_summary = group_subject_summary.groupby('trial_type').agg(
    n_subjects_RT=('mean_RT_ms', 'count'), mean_RT_ms=('mean_RT_ms', 'mean'), SEM_RT_ms=('mean_RT_ms', 'sem'),
    n_subjects_accuracy=('accuracy_pct', 'count'), accuracy_pct=('accuracy_pct', 'mean'), SEM_accuracy_pct=('accuracy_pct', 'sem'),
).reindex(TRIAL_TYPES)

display(run_summary.head())
display(subject_summary.head())

## 4. Plot helpers

The pooled bar is the mean calculated from all valid trials in the table.
Optional black points/lines show run-level values, so variability is visible instead of hidden by one bar.

In [ ]:
def draw_bars(ax, table, metric, title, run_points=None, subject_points=None, sem_column=None):
    table = table.set_index('trial_type').reindex(TRIAL_TYPES) if 'trial_type' in table else table.reindex(TRIAL_TYPES)
    x = np.arange(len(TRIAL_TYPES))
    values = table[metric].to_numpy(dtype=float)
    ax.bar(x, values, color=COLORS, width=0.65, alpha=0.85)

    if sem_column is not None:
        errors = table[sem_column].to_numpy(dtype=float)
        valid = np.isfinite(values) & np.isfinite(errors)
        ax.errorbar(x[valid], values[valid], yerr=errors[valid], fmt='none', color='black', capsize=4)

    if subject_points is not None:
        for i, condition in enumerate(TRIAL_TYPES):
            y = subject_points.loc[subject_points['trial_type'].eq(condition), metric].dropna().to_numpy()
            offset = np.linspace(-0.16, 0.16, len(y)) if len(y) > 1 else np.zeros(len(y))
            ax.scatter(i + offset, y, color='black', s=22, alpha=0.55, zorder=3)

    if run_points is not None and not run_points.empty:
        wide = run_points.pivot(index='run', columns='trial_type', values=metric).reindex(columns=TRIAL_TYPES)
        for y in wide.to_numpy(dtype=float):
            valid = np.isfinite(y)
            if valid.any():
                ax.plot(x[valid], y[valid], 'o-', color='black', markersize=4, linewidth=1, alpha=0.4, zorder=4)

    for i, value in enumerate(values):
        if np.isfinite(value):
            ax.annotate(f'{value:.1f}', (i, value), xytext=(0, 6), textcoords='offset points', ha='center')
        else:
            ax.text(i, 0.02, 'No data', transform=ax.get_xaxis_transform(), ha='center')
    ax.set_xticks(x, TRIAL_TYPES)
    ax.set_title(title)
    ax.set_ylabel(dict(METRICS)[metric])
    ax.grid(axis='y', alpha=0.15)
    ax.set_axisbelow(True)
    ax.set_ylim(bottom=0)
    if metric == 'accuracy_pct':
        ax.set_ylim(0, 112)
        ax.set_yticks(range(0, 101, 20))


def plot_pair(table, title, run_points=None, subject_points=None, group=False):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), layout='constrained')
    for ax, (metric, label) in zip(axes, METRICS):
        sem = {'mean_RT_ms': 'SEM_RT_ms', 'accuracy_pct': 'SEM_accuracy_pct'}[metric] if group else None
        draw_bars(ax, table, metric, label, run_points, subject_points, sem)
    fig.suptitle(title)
    return fig


figures = {}

## 5. Selected subject: pooled across included runs

In [ ]:
selected = subject_summary.loc[subject_summary['subject'].eq(SELECTED_SUBJECT)]
selected_runs = run_summary.loc[run_summary['subject'].eq(SELECTED_SUBJECT)]
if selected.empty:
    print(f'No data for subject {SELECTED_SUBJECT}.')
else:
    display(selected.round(2))
    figures['selected_subject'] = plot_pair(
        selected, f'{SELECTED_SUBJECT} | pooled included runs', run_points=selected_runs
    )
    plt.show()

## 6. All subjects: one panel per subject

In [ ]:
def plot_subject_panels(summary, metric):
    subjects = sorted(summary['subject'].unique())
    columns = max(1, min(PANEL_COLUMNS, len(subjects)))
    rows = int(np.ceil(len(subjects) / columns))
    fig, axes = plt.subplots(rows, columns, figsize=(4.2 * columns, 3.6 * rows), squeeze=False, layout='constrained')
    run_counts = coverage.set_index('subject')['n_runs']
    for ax, subject in zip(axes.flat, subjects):
        table = summary.loc[summary['subject'].eq(subject)]
        runs = run_summary.loc[run_summary['subject'].eq(subject)]
        draw_bars(ax, table, metric, f'{subject} | {run_counts[subject]}/12 runs', run_points=runs)
    for ax in axes.flat[len(subjects):]:
        ax.set_visible(False)
    fig.suptitle(f'All subjects | {dict(METRICS)[metric]}')
    return fig


for metric, label in METRICS:
    figures[f'all_subjects_{metric}'] = plot_subject_panels(subject_summary, metric)
    plt.show()

## 7. Group mean and paired condition differences

In [ ]:
if group_subject_summary.empty:
    print('No subjects meet the group inclusion rule.')
else:
    display(group_summary.round(2))
    n_subjects = group_subject_summary['subject'].nunique()
    figures['group_mean'] = plot_pair(group_summary.reset_index(), f'Group mean | {n_subjects} subjects', subject_points=group_subject_summary, group=True)
    plt.show()

    # Difference = EDS minus the comparison condition. Positive RT means slower EDS.
    paired = group_subject_summary.pivot(index='subject', columns='trial_type', values=['mean_RT_ms', 'accuracy_pct'])
    contrasts = {'EDS - IDS': ('EDS', 'IDS'), 'EDS - Stay': ('EDS', 'Stay')}
    difference_rows = []
    for metric, label in METRICS:
        for name, (left, right) in contrasts.items():
            values = paired[(metric, left)] - paired[(metric, right)]
            difference_rows.append({'metric': label, 'contrast': name, 'mean_difference': values.mean(), 'SEM': values.sem(), 'n_positive': (values > 0).sum(), 'n_negative': (values < 0).sum()})
    display(pd.DataFrame(difference_rows).round(2))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), layout='constrained')
    for ax, (metric, label) in zip(axes, METRICS):
        means = [ (paired[(metric, a)] - paired[(metric, b)]).mean() for a, b in contrasts.values() ]
        errors = [ (paired[(metric, a)] - paired[(metric, b)]).sem() for a, b in contrasts.values() ]
        ax.bar(range(len(contrasts)), means, yerr=errors, color=['#4878D0', '#6ACC64'], capsize=4)
        ax.axhline(0, color='black', linewidth=1)
        ax.set_xticks(range(len(contrasts)), contrasts)
        ax.set_ylabel(f'{label} difference')
        ax.set_title(label)
        ax.grid(axis='y', alpha=0.15)
    figures['paired_differences'] = fig
    plt.show()

## 8. Run-by-run trends

In [ ]:
selected_runs = run_summary.loc[run_summary['subject'].eq(SELECTED_SUBJECT)]
if selected_runs.empty:
    print(f'No included runs for subject {SELECTED_SUBJECT}.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4), layout='constrained')
    wide = selected_runs.pivot(index='run', columns='trial_type')
    for ax, (metric, label) in zip(axes, METRICS):
        for condition, color in zip(TRIAL_TYPES, COLORS):
            ax.plot(wide.index, wide[metric, condition], 'o-', color=color, label=condition)
        ax.set(xlabel='Run', ylabel=label, xticks=RUNS)
        ax.set_ylim(bottom=0)
        if metric == 'accuracy_pct':
            ax.set_ylim(0, 105)
        ax.grid(axis='y', alpha=0.15)
        ax.legend()
    fig.suptitle(f'{SELECTED_SUBJECT} | run-by-run trends')
    figures['run_trends'] = fig
    plt.show()

## 9. Optional export

Set `SAVE_RESULTS = True` in the Settings cell to save the tables and figures.

In [ ]:
if SAVE_RESULTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    tables = {
        'file_manifest': manifest,
        'run_coverage': coverage,
        'run_summary': run_summary,
        'subject_summary': subject_summary,
        'group_summary': group_summary.reset_index(),
    }
    for name, table in tables.items():
        table.to_csv(OUTPUT_DIR / f'{name}.csv', index=False)
    for name, figure in figures.items():
        figure.savefig(OUTPUT_DIR / f'{name}.png', dpi=180, bbox_inches='tight')
    print(f'Saved results to {OUTPUT_DIR.resolve()}')

## 10. Linear mixed-effects model for the ordered condition effect

The conditions are coded in the hypothesized order `EDS = 1`, `IDS = 2`, `Stay = 3`. The coefficient for `condition_order` tests the linear trend across this order: a negative coefficient means RT decreases from EDS through IDS to Stay. `run` is included as a fixed-effect covariate, and each subject has a random intercept.

In [ ]:
import statsmodels.formula.api as smf

lme_data = data.loc[data['RT_ms'].notna(), ['subject', 'run', 'trial_type', 'RT_ms']].copy()
lme_data['subject'] = lme_data['subject'].astype(str)
lme_data['condition_order'] = lme_data['trial_type'].map({'EDS': 1, 'IDS': 2, 'Stay': 3}).astype(float)

if lme_data['subject'].nunique() < 2:
    print('LME requires at least two subjects with valid RT data.')
else:
    lme_model = smf.mixedlm(
        'RT_ms ~ condition_order + run',
        data=lme_data,
        groups='subject',
        re_formula='1',
    )
    lme_result = lme_model.fit(reml=True, method='powell')
    lme_fixed_effects = pd.DataFrame({
        'coefficient': lme_result.fe_params,
        'SE': lme_result.bse_fe,
        'z': lme_result.tvalues[lme_result.fe_params.index],
        'p_value': lme_result.pvalues[lme_result.fe_params.index],
    })
    lme_fixed_effects[['CI_low', 'CI_high']] = lme_result.conf_int().loc[lme_result.fe_params.index].to_numpy()
    print(lme_result.summary())
    display(lme_fixed_effects.round(3))

In [ ]:
if 'lme_result' not in globals():
    print('Run the LME cell first.')
else:
    from patsy import dmatrix

    condition_labels = ['EDS', 'IDS', 'Stay']
    condition_order = [1, 2, 3]
    prediction_data = pd.DataFrame({
        'condition_order': condition_order,
        'run': lme_data['run'].mean(),
    })
    design_matrix = dmatrix(
        'condition_order + run',
        prediction_data,
        return_type='dataframe',
    )
    fixed_names = lme_result.fe_params.index
    fixed_covariance = lme_result.cov_params().loc[fixed_names, fixed_names].to_numpy()
    predicted = np.asarray(lme_result.predict(prediction_data), dtype=float)
    prediction_se = np.sqrt(np.einsum('ij,jk,ik->i', design_matrix, fixed_covariance, design_matrix))
    prediction_data['estimate'] = predicted
    prediction_data['CI_low'] = predicted - 1.96 * prediction_se
    prediction_data['CI_high'] = predicted + 1.96 * prediction_se

    observed_subject_means = lme_data.groupby(['subject', 'trial_type'], observed=True)['RT_ms'].mean().reset_index()
    observed_subject_means['condition_order'] = observed_subject_means['trial_type'].map(
        {'EDS': 1, 'IDS': 2, 'Stay': 3}
    )

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), layout='constrained')
    x = np.asarray(condition_order)
    axes[0].plot(x, prediction_data['estimate'], color='#1F4E79', linewidth=2, marker='o', markersize=7)
    axes[0].fill_between(
        x,
        prediction_data['CI_low'],
        prediction_data['CI_high'],
        color='#1F4E79',
        alpha=0.18,
        label='95% CI',
    )
    axes[0].set_xticks(x, condition_labels)
    axes[0].set_title('LME linear trend')
    axes[0].set_ylabel('Predicted reaction time (ms)')
    axes[0].grid(axis='y', alpha=0.2)
    axes[0].set_axisbelow(True)

    for subject, subject_data in observed_subject_means.groupby('subject'):
        axes[1].plot(
            subject_data['condition_order'],
            subject_data['RT_ms'],
            'o-',
            color='#D95F02',
            alpha=0.45,
            linewidth=1,
            markersize=4,
        )
    observed_means = observed_subject_means.groupby('condition_order')['RT_ms'].mean().reindex(condition_order)
    axes[1].plot(x, observed_means, 'o-', color='black', linewidth=2, markersize=8, label='Subject mean')
    axes[1].set_xticks(x, condition_labels)
    axes[1].set_title('Observed subject means')
    axes[1].set_ylabel('Mean reaction time (ms)')
    axes[1].grid(axis='y', alpha=0.2)
    axes[1].set_axisbelow(True)
    axes[1].legend()
    fig.suptitle('Ordered condition effect: EDS -> IDS -> Stay')
    if 'figures' not in globals():
        figures = {}
    figures['lme_reaction_time'] = fig
    plt.show()

In [ ]:
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

subjects = sorted(trial_rt_data['subject'].unique())
columns = max(1, min(PANEL_COLUMNS, len(subjects)))
rows = int(np.ceil(len(subjects) / columns))
fig, axes = plt.subplots(
    rows,
    columns,
    figsize=(5.0 * columns, 4.5 * rows),
    squeeze=False,
    layout='constrained',
)

for ax, subject in zip(axes.flat, subjects):
    subject_trials = trial_rt_data.loc[trial_rt_data['subject'].eq(subject)]
    subject_runs = run_rt_data.loc[run_rt_data['subject'].eq(subject)]
    distributions = [
        subject_trials.loc[subject_trials['trial_type'].eq(condition), 'RT_ms'].to_numpy()
        for condition in TRIAL_TYPES
    ]
    x = np.arange(1, len(TRIAL_TYPES) + 1)
    violin = ax.violinplot(
        distributions,
        positions=x,
        widths=0.72,
        showextrema=False,
        showmeans=False,
    )
    for body, color in zip(violin['bodies'], COLORS):
        body.set_facecolor(color)
        body.set_edgecolor(color)
        body.set_alpha(0.6)

    condition_summary = subject_trials.groupby('trial_type', observed=True)['RT_ms'].agg(
        mean='mean',
        sem='sem',
    ).reindex(TRIAL_TYPES)
    ax.errorbar(
        x,
        condition_summary['mean'],
        yerr=condition_summary['sem'],
        fmt='D',
        color='black',
        markersize=5,
        capsize=4,
        linewidth=1.2,
        label='Mean +/- SEM' if subject == subjects[0] else None,
        zorder=4,
    )

    for i, condition in enumerate(TRIAL_TYPES, start=1):
        trial_values = subject_trials.loc[
            subject_trials['trial_type'].eq(condition), 'RT_ms'
        ].to_numpy()
        ax.scatter(
            np.full(len(trial_values), i),
            trial_values,
            color=COLORS[i - 1],
            alpha=0.16,
            s=8,
            edgecolors='none',
            zorder=2,
        )
        run_values = subject_runs.loc[
            subject_runs['trial_type'].eq(condition), 'mean_RT_ms'
        ].to_numpy()
        ax.scatter(
            np.full(len(run_values), i),
            run_values,
            color='black',
            s=42,
            alpha=0.9,
            edgecolors='white',
            linewidths=0.5,
            label='Run mean' if subject == subjects[0] and i == 1 else None,
            zorder=5,
        )

    ax.set_xticks(x, TRIAL_TYPES)
    ax.set_xlabel('Condition')
    ax.set_ylabel('Reaction time (ms)')
    ax.set_title(str(subject))
    ax.grid(axis='y', alpha=0.2)
    ax.set_axisbelow(True)

for ax in axes.flat[len(subjects):]:
    ax.set_visible(False)
axes.flat[0].legend()
fig.suptitle('Reaction time distribution by condition for each subject')
figures['subject_violin_rt_by_condition'] = fig
plt.show()

In [ ]:
group_condition_summary = trial_rt_data.groupby('trial_type', observed=True)['RT_ms'].agg(
    mean='mean',
    sem='sem',
).reindex(TRIAL_TYPES)
distributions = [
    trial_rt_data.loc[trial_rt_data['trial_type'].eq(condition), 'RT_ms'].to_numpy()
    for condition in TRIAL_TYPES
]

fig, ax = plt.subplots(figsize=(8, 5), layout='constrained')
x = np.arange(1, len(TRIAL_TYPES) + 1)
violin = ax.violinplot(
    distributions,
    positions=x,
    widths=0.72,
    showextrema=False,
    showmeans=False,
)
for body, color in zip(violin['bodies'], COLORS):
    body.set_facecolor(color)
    body.set_edgecolor(color)
    body.set_alpha(0.6)

ax.errorbar(
    x,
    group_condition_summary['mean'],
    yerr=group_condition_summary['sem'],
    fmt='D',
    color='black',
    markersize=6,
    capsize=5,
    linewidth=1.5,
    label='Mean +/- SEM',
    zorder=4,
)
for i, condition in enumerate(TRIAL_TYPES, start=1):
    trial_values = trial_rt_data.loc[
        trial_rt_data['trial_type'].eq(condition), 'RT_ms'
    ].to_numpy()
    ax.scatter(
        np.full(len(trial_values), i),
        trial_values,
        color=COLORS[i - 1],
        alpha=0.16,
        s=10,
        edgecolors='none',
        zorder=2,
    )
    run_values = run_rt_data.loc[
        run_rt_data['trial_type'].eq(condition), 'mean_RT_ms'
    ].to_numpy()
    ax.scatter(
        np.full(len(run_values), i),
        run_values,
        color='black',
        s=52,
        alpha=0.9,
        edgecolors='white',
        linewidths=0.6,
        label='Run mean' if i == 1 else None,
        zorder=5,
    )

ax.set_xticks(x, TRIAL_TYPES)
ax.set_xlabel('Condition')
ax.set_ylabel('Reaction time (ms)')
ax.set_title('Group reaction time distribution by condition')
ax.grid(axis='y', alpha=0.2)
ax.set_axisbelow(True)
ax.legend()
figures['group_violin_rt_by_condition'] = fig
plt.show()